In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder \
.appName("E-CommerceAnalytics")\
.getOrCreate()

In [5]:
raw_customers = [
("C001","Rahul","29","Bangalore","Electronics,Fashion"),
("C002","Sneha","Thirty Two","Delhi","Fashion"),
("C003","Aman",None,"Mumbai",["Home","Electronics"]),
("C004","Pallavi","27","Pune","Electronics|Beauty"),

("C005","", "35","Chennai",None)
]

In [4]:
raw_sellers = [
("S001","TechWorld","Electronics","2019-06-01"),
("S002","FashionHub","Fashion","01/07/2020"),
("S003","HomeEssentials","Home","2018/09/15"),
("S004","BeautyStore","Beauty","invalid_date")
]


In [3]:
raw_products = [
("P001","Laptop","Electronics","S001","55000"),
("P002","Headphones","Electronics","S001","2500"),
("P003","T-Shirt","Fashion","S002","1200"),
("P004","Sofa","Home","S003","45000"),
("P005","Face Cream","Beauty","S004","800")
]

In [ ]:
raw_products = [
("P001","Laptop","Electronics","S001","55000"),
("P002","Headphones","Electronics","S001","2500"),
("P003","T-Shirt","Fashion","S002","1200"),
("P004","Sofa","Home","S003","45000"),
("P005","Face Cream","Beauty","S004","800")
]

In [6]:
raw_activity = [
("C001","search,view,add_to_cart","{'device':'mobile'}",180),
("C002",["search","view"],"device=laptop",90),
("C003","search|view|purchase",None,120),
("C004",None,"{'device':'tablet'}",60),
("C005","search","{'device':'mobile'}",30)
]

PART A

In [7]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
customer_schema = StructType([
StructField("customer_id", StringType(), True),
StructField("name", StringType(), True),
StructField("age", StringType(), True),
StructField("city", StringType(), True),
StructField("interests", StringType(), True)
])
raw_customers_df = spark.createDataFrame(raw_customers, schema=customer_schema)


In [8]:
seller_schema = StructType([
StructField("seller_id", StringType(), True),
StructField("seller_name", StringType(), True),
StructField("category", StringType(), True),
StructField("start_date", StringType(), True)
])
raw_sellers_df = spark.createDataFrame(raw_sellers, schema=seller_schema)


In [9]:
product_schema = StructType([
StructField("product_id", StringType(), True),
StructField("product", StringType(), True),
StructField("category", StringType(), True),
StructField("seller_id", StringType(), True),
StructField("start_date", StringType(), True)
])
raw_products_df = spark.createDataFrame(raw_products, schema=product_schema)


In [11]:
raw_orders = [
    ("O001", "C001", "P001", "2023-01-05", "Delivered", "55000"),
    ("O002", "C002", "P003", "2023-01-06", "Pending", "1200"),
    ("O003", "C001", "P002", "2023-01-07", "Delivered", "2500"),
    ("O004", "C003", "P004", "2023-01-08", "Shipped", "45000"),
    ("O005", "C005", "P005", "2023-01-09", "Delivered", "800")
]
order_schema = StructType([
StructField("order_id", StringType(), True),
StructField("customer_id", StringType(), True),
StructField("product_id", StringType(), True),
StructField("order_date", StringType(), True),
StructField("status", StringType(), True),
StructField("amount", StringType(), True)
])
raw_orders_df = spark.createDataFrame(raw_orders, schema=order_schema)

In [12]:
activity_schema = StructType([
StructField("customer_id", StringType(), True),
StructField("actions", StringType(), True),
StructField("metadata", StringType(), True),
StructField("duration", IntegerType(), True)
])
raw_activity_df = spark.createDataFrame(raw_activity, schema=activity_schema)


In [13]:
from pyspark.sql.functions import regexp_extract, col, when

customers_df = raw_customers_df \
    .withColumn("age", when(regexp_extract("age", "\d+", 0) == "", None)
                .otherwise(regexp_extract("age", "\d+", 0)).cast("int")) \
    .withColumn("name", when(col("name") == "", None).otherwise(col("name")))

customers_df.show()


<>:4: SyntaxWarning: invalid escape sequence '\d'
<>:5: SyntaxWarning: invalid escape sequence '\d'
<>:4: SyntaxWarning: invalid escape sequence '\d'
<>:5: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipython-input-426912395.py:4: SyntaxWarning: invalid escape sequence '\d'
  .withColumn("age", when(regexp_extract("age", "\d+", 0) == "", None)
/tmp/ipython-input-426912395.py:5: SyntaxWarning: invalid escape sequence '\d'
  .otherwise(regexp_extract("age", "\d+", 0)).cast("int")) \


+-----------+-------+----+---------+-------------------+
|customer_id|   name| age|     city|          interests|
+-----------+-------+----+---------+-------------------+
|       C001|  Rahul|  29|Bangalore|Electronics,Fashion|
|       C002|  Sneha|NULL|    Delhi|            Fashion|
|       C003|   Aman|NULL|   Mumbai|[Home, Electronics]|
|       C004|Pallavi|  27|     Pune| Electronics|Beauty|
|       C005|   NULL|  35|  Chennai|               NULL|
+-----------+-------+----+---------+-------------------+



In [14]:
from pyspark.sql.functions import split, regexp_replace

customers_df = customers_df.withColumn(
    "interests",
    split(regexp_replace("interests", "[|]", ","), ",")
)



from pyspark.sql.functions import split, regexp_replace

activity_df = raw_activity_df.withColumn(
    "actions",
    split(regexp_replace("actions", "[|]", ","), ",")
)

In [18]:
from pyspark.sql.functions import col, to_date, coalesce, split, lit, array_remove, try_to_timestamp

# Make an empty string array: split("", ",") -> [""] then remove "" -> []
empty_string_array = array_remove(split(lit(""), ","), "")

customers_df = customers_df.withColumn(
    "interests",
    coalesce(col("interests"), empty_string_array)
)

orders_df = raw_orders_df.filter(col("order_date").isNotNull())

sellers_df = raw_sellers_df.withColumn(
    "start_date",
    coalesce(
        to_date(try_to_timestamp(col("start_date"), lit("yyyy-MM-dd"))),
        to_date(try_to_timestamp(col("start_date"), lit("dd/MM/yyyy"))),
        to_date(try_to_timestamp(col("start_date"), lit("yyyy/MM/dd")))
    )
)

In [20]:
products_df = raw_products_df
customers_df.printSchema()
customers_df.show()
sellers_df.printSchema()
sellers_df.show()
products_df.printSchema()
products_df.show()
orders_df.printSchema()
orders_df.show()
activity_df.printSchema()
activity_df.show()

root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- interests: array (nullable = false)
 |    |-- element: string (containsNull = false)

+-----------+-------+----+---------+--------------------+
|customer_id|   name| age|     city|           interests|
+-----------+-------+----+---------+--------------------+
|       C001|  Rahul|  29|Bangalore|[Electronics, Fas...|
|       C002|  Sneha|NULL|    Delhi|           [Fashion]|
|       C003|   Aman|NULL|   Mumbai|[[Home,  Electron...|
|       C004|Pallavi|  27|     Pune|[Electronics, Bea...|
|       C005|   NULL|  35|  Chennai|                  []|
+-----------+-------+----+---------+--------------------+

root
 |-- seller_id: string (nullable = true)
 |-- seller_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- start_date: date (nullable = true)

+---------+--------------+-----------+----------+
|seller_id

PART B

In [21]:
orders_products_df  = orders_df.join(products_df, "product_id", "inner")
orders_products_df.show()

+----------+--------+-----------+----------+---------+------+----------+-----------+---------+----------+
|product_id|order_id|customer_id|order_date|   status|amount|   product|   category|seller_id|start_date|
+----------+--------+-----------+----------+---------+------+----------+-----------+---------+----------+
|      P001|    O001|       C001|2023-01-05|Delivered| 55000|    Laptop|Electronics|     S001|     55000|
|      P002|    O003|       C001|2023-01-07|Delivered|  2500|Headphones|Electronics|     S001|      2500|
|      P003|    O002|       C002|2023-01-06|  Pending|  1200|   T-Shirt|    Fashion|     S002|      1200|
|      P004|    O004|       C003|2023-01-08|  Shipped| 45000|      Sofa|       Home|     S003|     45000|
|      P005|    O005|       C005|2023-01-09|Delivered|   800|Face Cream|     Beauty|     S004|       800|
+----------+--------+-----------+----------+---------+------+----------+-----------+---------+----------+



In [22]:
products_seller_df  = products_df.join(broadcast(sellers_df), "seller_id", "inner")
products_seller_df.show()


+---------+----------+----------+-----------+----------+--------------+-----------+----------+
|seller_id|product_id|   product|   category|start_date|   seller_name|   category|start_date|
+---------+----------+----------+-----------+----------+--------------+-----------+----------+
|     S001|      P001|    Laptop|Electronics|     55000|     TechWorld|Electronics|2019-06-01|
|     S001|      P002|Headphones|Electronics|      2500|     TechWorld|Electronics|2019-06-01|
|     S002|      P003|   T-Shirt|    Fashion|      1200|    FashionHub|    Fashion|2020-07-01|
|     S003|      P004|      Sofa|       Home|     45000|HomeEssentials|       Home|2018-09-15|
|     S004|      P005|Face Cream|     Beauty|       800|   BeautyStore|     Beauty|      NULL|
+---------+----------+----------+-----------+----------+--------------+-----------+----------+



In [23]:
orders_customers_df  = orders_df.join(customers_df, "customer_id", "inner")
orders_customers_df.show()


+-----------+--------+----------+----------+---------+------+-----+----+---------+--------------------+
|customer_id|order_id|product_id|order_date|   status|amount| name| age|     city|           interests|
+-----------+--------+----------+----------+---------+------+-----+----+---------+--------------------+
|       C001|    O001|      P001|2023-01-05|Delivered| 55000|Rahul|  29|Bangalore|[Electronics, Fas...|
|       C001|    O003|      P002|2023-01-07|Delivered|  2500|Rahul|  29|Bangalore|[Electronics, Fas...|
|       C002|    O002|      P003|2023-01-06|  Pending|  1200|Sneha|NULL|    Delhi|           [Fashion]|
|       C003|    O004|      P004|2023-01-08|  Shipped| 45000| Aman|NULL|   Mumbai|[[Home,  Electron...|
|       C005|    O005|      P005|2023-01-09|Delivered|   800| NULL|  35|  Chennai|                  []|
+-----------+--------+----------+----------+---------+------+-----+----+---------+--------------------+



In [24]:
from pyspark.sql.functions import broadcast

# Join orders_df with broadcasted customers_df
orders_customers_broadcast_df = orders_df.join(broadcast(customers_df), "customer_id", "inner")

print("Physical plan for join with broadcasted customers_df:")
orders_customers_broadcast_df.explain(True)

Physical plan for join with broadcasted customers_df:
== Parsed Logical Plan ==
'Join UsingJoin(Inner, [customer_id])
:- Filter isnotnull(order_date#17)
:  +- LogicalRDD [order_id#14, customer_id#15, product_id#16, order_date#17, status#18, amount#19], false
+- ResolvedHint (strategy=broadcast)
   +- Project [customer_id#0, name#25, age#24, city#3, coalesce(interests#45, array_remove(split(, ,, -1), )) AS interests#46]
      +- Project [customer_id#0, name#25, age#24, city#3, coalesce(interests#44, array_remove(split(, ,, -1), )) AS interests#45]
         +- Project [customer_id#0, name#25, age#24, city#3, coalesce(interests#42, array_remove(split(, ,, -1), )) AS interests#44]
            +- Project [customer_id#0, name#25, age#24, city#3, split(regexp_replace(interests#4, [|], ,, 1), ,, -1) AS interests#42]
               +- Project [customer_id#0, CASE WHEN (name#1 = ) THEN cast(null as string) ELSE name#1 END AS name#25, age#24, city#3, interests#4]
                  +- Project [cus

In [25]:
from pyspark.sql.functions import broadcast

# Join orders_df with broadcasted customers_df
orders_customers_broadcast_df = orders_df.join(broadcast(customers_df), "customer_id", "inner")

print("Physical plan for join with broadcasted customers_df:")
orders_customers_broadcast_df.explain(True)


Physical plan for join with broadcasted customers_df:
== Parsed Logical Plan ==
'Join UsingJoin(Inner, [customer_id])
:- Filter isnotnull(order_date#17)
:  +- LogicalRDD [order_id#14, customer_id#15, product_id#16, order_date#17, status#18, amount#19], false
+- ResolvedHint (strategy=broadcast)
   +- Project [customer_id#0, name#25, age#24, city#3, coalesce(interests#45, array_remove(split(, ,, -1), )) AS interests#46]
      +- Project [customer_id#0, name#25, age#24, city#3, coalesce(interests#44, array_remove(split(, ,, -1), )) AS interests#45]
         +- Project [customer_id#0, name#25, age#24, city#3, coalesce(interests#42, array_remove(split(, ,, -1), )) AS interests#44]
            +- Project [customer_id#0, name#25, age#24, city#3, split(regexp_replace(interests#4, [|], ,, 1), ,, -1) AS interests#42]
               +- Project [customer_id#0, CASE WHEN (name#1 = ) THEN cast(null as string) ELSE name#1 END AS name#25, age#24, city#3, interests#4]
                  +- Project [cus

In [26]:
# 1. Eliminate customers without any orders
customers_with_orders_df = customers_df.join(orders_df, "customer_id", "left_semi")
orphan_customers_df = customers_df.join(orders_df, "customer_id", "left_anti")

print("Orphan Customers:")
orphan_customers_df.show()

# Update customers_df to only include customers with orders
customers_df = customers_with_orders_df

# 2. Eliminate products without any orders
products_with_orders_df = products_df.join(orders_df, "product_id", "left_semi")
orphan_products_df = products_df.join(orders_df, "product_id", "left_anti")

print("Orphan Products:")
orphan_products_df.show()

# Update products_df to only include products with orders
products_df = products_with_orders_df

# 3. Eliminate sellers without any products
sellers_with_products_df = sellers_df.join(products_df, "seller_id", "left_semi")
orphan_sellers_df = sellers_df.join(products_df, "seller_id", "left_anti")

print("Orphan Sellers:")
orphan_sellers_df.show()

# Update sellers_df to only include sellers with products
sellers_df = sellers_with_products_df

print("DataFrames after eliminating orphan records:")
customers_df.show()
products_df.show()
sellers_df.show()

Orphan Customers:
+-----------+-------+---+----+--------------------+
|customer_id|   name|age|city|           interests|
+-----------+-------+---+----+--------------------+
|       C004|Pallavi| 27|Pune|[Electronics, Bea...|
+-----------+-------+---+----+--------------------+

Orphan Products:
+----------+-------+--------+---------+----------+
|product_id|product|category|seller_id|start_date|
+----------+-------+--------+---------+----------+
+----------+-------+--------+---------+----------+

Orphan Sellers:
+---------+-----------+--------+----------+
|seller_id|seller_name|category|start_date|
+---------+-----------+--------+----------+
+---------+-----------+--------+----------+

DataFrames after eliminating orphan records:
+-----------+-----+----+---------+--------------------+
|customer_id| name| age|     city|           interests|
+-----------+-----+----+---------+--------------------+
|       C001|Rahul|  29|Bangalore|[Electronics, Fas...|
|       C002|Sneha|NULL|    Delhi|   

PART C

In [27]:
revenue_category_df = orders_products_df.groupBy("category").agg(sum("amount").alias("total_revenue"))
revenue_category_df.show()


+-----------+-------------+
|   category|total_revenue|
+-----------+-------------+
|       Home|      45000.0|
|    Fashion|       1200.0|
|Electronics|      57500.0|
|     Beauty|        800.0|
+-----------+-------------+



In [28]:
revenue_seller_df = orders_products_df.groupBy("seller_id").agg(sum("amount").alias("total_revenue"))
revenue_seller_df.show()


+---------+-------------+
|seller_id|total_revenue|
+---------+-------------+
|     S004|        800.0|
|     S001|      57500.0|
|     S002|       1200.0|
|     S003|      45000.0|
+---------+-------------+



In [29]:
orders_customers_df = orders_df.groupBy("customer_id").agg(count("order_id").alias("total_orders"))
orders_customers_df.show()


+-----------+------------+
|customer_id|total_orders|
+-----------+------------+
|       C001|           2|
|       C002|           1|
|       C003|           1|
|       C005|           1|
+-----------+------------+



In [30]:
average_order_value_df = orders_df.withColumn("amount", col("amount").cast("double")) \
    .groupBy("customer_id") \
    .agg(avg("amount").alias("average_order_value"))
average_order_value_df.show()

+-----------+-------------------+
|customer_id|average_order_value|
+-----------+-------------------+
|       C001|            28750.0|
|       C002|             1200.0|
|       C003|            45000.0|
|       C005|              800.0|
+-----------+-------------------+



In [31]:
zero_delivery_sellers_df = orders_products_df.filter(col("status") == "Delivered")
zero_delivery_sellers_df.show()


+----------+--------+-----------+----------+---------+------+----------+-----------+---------+----------+
|product_id|order_id|customer_id|order_date|   status|amount|   product|   category|seller_id|start_date|
+----------+--------+-----------+----------+---------+------+----------+-----------+---------+----------+
|      P001|    O001|       C001|2023-01-05|Delivered| 55000|    Laptop|Electronics|     S001|     55000|
|      P002|    O003|       C001|2023-01-07|Delivered|  2500|Headphones|Electronics|     S001|      2500|
|      P005|    O005|       C005|2023-01-09|Delivered|   800|Face Cream|     Beauty|     S004|       800|
+----------+--------+-----------+----------+---------+------+----------+-----------+---------+----------+



PART D

In [32]:
from pyspark.sql.window import Window
from pyspark.sql.functions import sum, col, rank

# Calculate total spend per customer
total_spend_per_customer_df = orders_products_df.groupBy("customer_id") \
    .agg(sum(col("amount").cast("double")).alias("total_spend"))

# Define a window specification to rank customers by total spend
window_spec = Window.orderBy(col("total_spend").desc())

# Apply the rank function
customer_spend_rank_df = total_spend_per_customer_df.withColumn("spend_rank", rank().over(window_spec))

customer_spend_rank_df.show()

+-----------+-----------+----------+
|customer_id|total_spend|spend_rank|
+-----------+-----------+----------+
|       C001|    57500.0|         1|
|       C003|    45000.0|         2|
|       C002|     1200.0|         3|
|       C005|      800.0|         4|
+-----------+-----------+----------+



In [33]:
from pyspark.sql.window import Window
from pyspark.sql.functions import sum, col, rank

revenue_per_seller_category_df = orders_products_df.groupBy("category", "seller_id") \
    .agg(sum(col("amount").cast("double")).alias("total_revenue"))

window_spec_category = Window.partitionBy("category").orderBy(col("total_revenue").desc())

seller_category_rank_df = revenue_per_seller_category_df.withColumn("category_rank", rank().over(window_spec_category))

seller_category_rank_df.show()

+-----------+---------+-------------+-------------+
|   category|seller_id|total_revenue|category_rank|
+-----------+---------+-------------+-------------+
|     Beauty|     S004|        800.0|            1|
|Electronics|     S001|      57500.0|            1|
|    Fashion|     S002|       1200.0|            1|
|       Home|     S003|      45000.0|            1|
+-----------+---------+-------------+-------------+



In [34]:
from pyspark.sql.window import Window
from pyspark.sql.functions import sum, col, asc

daily_revenue_df = orders_products_df.withColumn("amount", col("amount").cast("double")) \
    .groupBy("order_date") \
    .agg(sum("amount").alias("daily_revenue"))


window_spec_daily = Window.orderBy(asc("order_date"))

running_revenue_df = daily_revenue_df.withColumn("running_revenue", sum("daily_revenue").over(window_spec_daily))

running_revenue_df.show()

+----------+-------------+---------------+
|order_date|daily_revenue|running_revenue|
+----------+-------------+---------------+
|2023-01-05|      55000.0|        55000.0|
|2023-01-06|       1200.0|        56200.0|
|2023-01-07|       2500.0|        58700.0|
|2023-01-08|      45000.0|       103700.0|
|2023-01-09|        800.0|       104500.0|
+----------+-------------+---------------+



In [35]:
from pyspark.sql.window import Window
from pyspark.sql.functions import sum, col, rank

product_revenue_per_category_df = orders_products_df.groupBy("category", "product_id", "product") \
    .agg(sum(col("amount").cast("double")).alias("product_revenue"))

window_spec_product_rank = Window.partitionBy("category").orderBy(col("product_revenue").desc())

top_2_products_per_category_df = product_revenue_per_category_df.withColumn("rank", rank().over(window_spec_product_rank)) \
    .filter(col("rank") <= 2)

top_2_products_per_category_df.show()

+-----------+----------+----------+---------------+----+
|   category|product_id|   product|product_revenue|rank|
+-----------+----------+----------+---------------+----+
|     Beauty|      P005|Face Cream|          800.0|   1|
|Electronics|      P001|    Laptop|        55000.0|   1|
|Electronics|      P002|Headphones|         2500.0|   2|
|    Fashion|      P003|   T-Shirt|         1200.0|   1|
|       Home|      P004|      Sofa|        45000.0|   1|
+-----------+----------+----------+---------------+----+



PART E

In [36]:
from pyspark.sql.functions import col, when
customer_spending_tiers_df = total_spend_per_customer_df.withColumn(
    "spending_tier",
    when(col("total_spend") > 10000, "High")
    .when((col("total_spend") > 1000) & (col("total_spend") <= 10000), "Medium")
    .otherwise("Low")
)

print("Customers classified into spending tiers:")
customer_spending_tiers_df.show()

Customers classified into spending tiers:
+-----------+-----------+-------------+
|customer_id|total_spend|spending_tier|
+-----------+-----------+-------------+
|       C003|    45000.0|         High|
|       C005|      800.0|          Low|
|       C001|    57500.0|         High|
|       C002|     1200.0|       Medium|
+-----------+-----------+-------------+



PART F

In [37]:
from pyspark.sql.functions import desc

sorted_categories_by_revenue_df = revenue_category_df.orderBy(desc("total_revenue"))
sorted_categories_by_revenue_df.show()


+-----------+-------------+
|   category|total_revenue|
+-----------+-------------+
|Electronics|      57500.0|
|       Home|      45000.0|
|    Fashion|       1200.0|
|     Beauty|        800.0|
+-----------+-------------+



In [38]:
from pyspark.sql.functions import col
sorted_sellers_by_category_revenue_df = seller_category_rank_df.orderBy(col("category").asc(), col("total_revenue").desc())

print("Sellers sorted by revenue within each category:")
sorted_sellers_by_category_revenue_df.show()


Sellers sorted by revenue within each category:
+-----------+---------+-------------+-------------+
|   category|seller_id|total_revenue|category_rank|
+-----------+---------+-------------+-------------+
|     Beauty|     S004|        800.0|            1|
|Electronics|     S001|      57500.0|            1|
|    Fashion|     S002|       1200.0|            1|
|       Home|     S003|      45000.0|            1|
+-----------+---------+-------------+-------------+



PART G

In [39]:
ordered_customers_df = orders_df.select("customer_id").distinct()
active_customers_df = activity_df.filter(col("actions").isNotNull()).select("customer_id").distinct()


In [40]:
active_customers_df.subtract(ordered_customers_df).show()

+-----------+
|customer_id|
+-----------+
+-----------+



In [41]:
active_customers_df.intersect(ordered_customers_df).show()


+-----------+
|customer_id|
+-----------+
|       C003|
|       C005|
|       C001|
|       C002|
+-----------+



In [42]:
print("\n--- Set Operations (Operating on rows) ---")
print("Customers active but never ordered (using subtract):")
active_customers_df.subtract(ordered_customers_df).show()

print("Customers who ordered AND were active (using intersect):")
active_customers_df.intersect(ordered_customers_df).show()

print("\n--- Join Operations (Operating on columns based on keys) ---")
print("Inner Join: Combining customer and order details for matching customer_ids:")
orders_df.join(customers_df, "customer_id", "inner").show()

print("Left Anti Join: Customers from 'customers_df' who are NOT in 'orders_df' (different from subtract, key-based):")
customers_df.join(orders_df, "customer_id", "left_anti").show()



--- Set Operations (Operating on rows) ---
Customers active but never ordered (using subtract):
+-----------+
|customer_id|
+-----------+
+-----------+

Customers who ordered AND were active (using intersect):
+-----------+
|customer_id|
+-----------+
|       C003|
|       C005|
|       C001|
|       C002|
+-----------+


--- Join Operations (Operating on columns based on keys) ---
Inner Join: Combining customer and order details for matching customer_ids:
+-----------+--------+----------+----------+---------+------+-----+----+---------+--------------------+
|customer_id|order_id|product_id|order_date|   status|amount| name| age|     city|           interests|
+-----------+--------+----------+----------+---------+------+-----+----+---------+--------------------+
|       C001|    O001|      P001|2023-01-05|Delivered| 55000|Rahul|  29|Bangalore|[Electronics, Fas...|
|       C002|    O002|      P003|2023-01-06|  Pending|  1200|Sneha|NULL|    Delhi|           [Fashion]|
|       C003|    O

PART H

In [43]:
products_seller_df.explain(True)
customer_spend_rank_df.explain(True)
sorted_categories_by_revenue_df.explain(True)

== Parsed Logical Plan ==
'Join UsingJoin(Inner, [seller_id])
:- LogicalRDD [product_id#9, product#10, category#11, seller_id#12, start_date#13], false
+- ResolvedHint (strategy=broadcast)
   +- Project [seller_id#5, seller_name#6, category#7, coalesce(to_date(try_to_timestamp(start_date#8, Some(yyyy-MM-dd), TimestampType, Some(Etc/UTC), false), None, Some(Etc/UTC), true), to_date(try_to_timestamp(start_date#8, Some(dd/MM/yyyy), TimestampType, Some(Etc/UTC), false), None, Some(Etc/UTC), true), to_date(try_to_timestamp(start_date#8, Some(yyyy/MM/dd), TimestampType, Some(Etc/UTC), false), None, Some(Etc/UTC), true)) AS start_date#47]
      +- LogicalRDD [seller_id#5, seller_name#6, category#7, start_date#8], false

== Analyzed Logical Plan ==
seller_id: string, product_id: string, product: string, category: string, start_date: string, seller_name: string, category: string, start_date: date
Project [seller_id#12, product_id#9, product#10, category#11, start_date#13, seller_name#6, categor

In [44]:
orders_products_df.cache()

DataFrame[product_id: string, order_id: string, customer_id: string, order_date: string, status: string, amount: string, product: string, category: string, seller_id: string, start_date: string]